# La memoria associativa di Hopfield

Il codice del capitolo [«La memoria associativa di Hopfield»](https://book.paithon.it/main/ModelliEnergia/memoria-associativa.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy scipy

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## La memoria associativa di Hopfield

[Leggi la pagina](https://book.paithon.it/main/ModelliEnergia/memoria-associativa.html)


### Una memoria che si ripara da sola, in poche righe


In [ ]:
import numpy as np

rng = np.random.default_rng(42)

# Tre lettere stilizzate 5x5: '#' = pixel acceso (+1), '.' = spento (-1)
LETTERE = {
    "T": ["#####",
          "..#..",
          "..#..",
          "..#..",
          "..#.."],
    "L": ["#....",
          "#....",
          "#....",
          "#....",
          "#####"],
    "X": ["#...#",
          ".#.#.",
          "..#..",
          ".#.#.",
          "#...#"],
}

def a_vettore(disegno):
    """Da lista di stringhe a vettore di +1/-1."""
    return np.array([1 if c == "#" else -1
                     for riga in disegno for c in riga])

def a_righe(s):
    """Da vettore di +1/-1 a cinque stringhe stampabili."""
    griglia = np.where(s == 1, "#", ".").reshape(5, 5)
    return ["".join(riga) for riga in griglia]

pattern = np.array([a_vettore(d) for d in LETTERE.values()])   # forma (3, 25)
N = pattern.shape[1]

# Regola di Hebb: somma dei prodotti esterni, diagonale a zero
W = (pattern.T @ pattern) / N
np.fill_diagonal(W, 0.0)

def energia(s):
    """E(s) = -1/2 s^T W s (soglie nulle)."""
    return -0.5 * s @ W @ s

def richiama(s, max_passate=10):
    """Aggiornamento asincrono fino a un punto fisso (minimo locale)."""
    s = s.copy()
    for _ in range(max_passate):
        cambiato = False
        for i in rng.permutation(N):        # un neurone alla volta
            campo = W[i] @ s                # campo locale sul neurone i
            # in caso di parita' il neurone resta com'e'. Il confronto vuole una
            # tolleranza: i legami sono multipli di 1/25, che in virgola mobile
            # non sono esatti, e un campo nullo esce come 1e-17 invece che come 0
            nuovo = np.sign(campo) if abs(campo) > 1e-9 else s[i]
            if nuovo != s[i]:
                s[i] = nuovo
                cambiato = True
        if not cambiato:                    # nessun cambiamento: stato stabile
            break
    return s

def corrompi(s, quanti=6):
    """Inverte 'quanti' bit scelti a caso (6 su 25 = 24%)."""
    s = s.copy()
    indici = rng.choice(N, size=quanti, replace=False)
    s[indici] = -s[indici]
    return s

for nome, disegno in LETTERE.items():
    originale = a_vettore(disegno)
    rumoroso = corrompi(originale)          # 24% dei pixel invertiti
    recuperato = richiama(rumoroso)
    esito = ("recuperato" if np.array_equal(recuperato, originale)
             else "NON recuperato")
    print(f"{nome}:  E = {energia(rumoroso):+.2f} "
          f"-> {energia(recuperato):+.2f}  ({esito})")
    print("   corrotto   richiamato")
    for r1, r2 in zip(a_righe(rumoroso), a_righe(recuperato)):
        print(f"   {r1}      {r2}")
    print()

## Alzare la temperatura: le macchine di Boltzmann

[Leggi la pagina](https://book.paithon.it/main/ModelliEnergia/boltzmann.html)


### Il sogno abbreviato: contrastive divergence


In [ ]:
import itertools
import numpy as np

# tre caselle e due taccuini; i legami esistono solo fra caselle e taccuini
W = np.array([[3.0, 3.0],     # la prima casella è legata ai due taccuini
              [3.0, 0.0],     # la seconda solo al primo
              [0.0, 3.0]])    # la terza solo al secondo
a, b = np.full(3, -2.0), np.full(2, -2.0)          # le soglie: spenti, da soli

# la distribuzione intera, stato per stato, con la sua Z
P = np.zeros((2,) * 5)
for s in itertools.product([0, 1], repeat=5):
    v, h = np.array(s[:3]), np.array(s[3:])
    P[s] = np.exp(a @ v + b @ h + v @ W @ h)
P /= P.sum()

def correlazione(Q):
    """Correlazione fra le due variabili di una tabella 2 x 2 di probabilità."""
    Q = Q / Q.sum()
    p, q = Q[1, :].sum(), Q[:, 1].sum()
    return (Q[1, 1] - p * q) / np.sqrt(p * (1 - p) * q * (1 - q))

tutte = list(itertools.product([0, 1], repeat=3))
ignorate = lambda assi: correlazione(P.sum(axis=assi))   # sommando via il resto
fissate = max(abs(correlazione(P[v])) for v in tutte)
print(f"i due taccuini, caselle fissate:  {fissate:.3f} nel caso peggiore")
print(f"i due taccuini, caselle ignorate: {ignorate((0, 1, 2)):.3f}")
fissati = max(abs(correlazione(P[:, :, :, i, j].sum(axis=2)))
              for i in (0, 1) for j in (0, 1))
print(f"caselle 1 e 2, taccuini fissati:  {fissati:.3f} nel caso peggiore")
print(f"caselle 1 e 2, taccuini ignorati: {ignorate((2, 3, 4)):.3f}")
print(f"caselle 2 e 3, taccuini ignorati: {ignorate((0, 3, 4)):.3f}")

## Oltre la partizione: tre modi di aggirare $Z$

[Leggi la pagina](https://book.paithon.it/main/ModelliEnergia/oltre-la-partizione.html)


### Prima via: campionare il paesaggio


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# Energia a doppia buca: minimi in x = -1 e x = +1, barriera in x = 0.
def energia(x):
    return (x**2 - 1.0)**2

def gradiente(x):            # dE/dx = 4x(x^2 - 1); lo score e' -gradiente
    return 4.0 * x * (x**2 - 1.0)

# Dinamica di Langevin: ventimila catene in parallelo, passi piccoli.
eps, passi, catene = 0.01, 2000, 20000
x = rng.normal(0.0, 2.0, size=catene)        # partenza qualsiasi
for _ in range(passi):
    x = x - 0.5 * eps * gradiente(x) + np.sqrt(eps) * rng.normal(size=catene)

# Verifica: p(x) = e^{-E(x)}/Z per quadratura numerica (si puo' fare in 1D).
griglia = np.linspace(-3, 3, 60001)
peso = np.exp(-energia(griglia))
Z = np.trapezoid(peso, griglia)
p_esatta = peso / Z

print(f"Z (quadratura)          = {Z:.4f}")
print(f"campioni |x| medio      = {np.abs(x).mean():.3f}")
print(f"esatto   |x| medio      = {np.trapezoid(np.abs(griglia)*p_esatta, griglia):.3f}")
print(f"frazione x>0 (campioni) = {(x > 0).mean():.3f}   (esatto 0.500)")
print()
print(" intervallo   campioni   esatto")
for a, b in [(-2.0, -1.5), (-1.5, -0.5), (-0.5, 0.5), (0.5, 1.5), (1.5, 2.0)]:
    emp = ((x >= a) & (x < b)).mean()
    m = (griglia >= a) & (griglia < b)
    ex = np.trapezoid(p_esatta[m], griglia[m])
    print(f" [{a:+.1f},{b:+.1f})    {emp:6.3f}   {ex:6.3f}")

In [ ]:
import numpy as np
from scipy import sparse
from scipy.sparse.linalg import eigs

energia = lambda x: (x**2 - 1.0)**2
gradiente = lambda x: 4.0 * x * (x**2 - 1.0)

# la distribuzione su cui la catena a passo eps si assesta: quella che la
# regola di transizione, scritta come matrice su una griglia fine, lascia ferma
griglia = np.linspace(-2.6, 2.6, 5201)
centro = (griglia >= -0.5) & (griglia < 0.5)
esatta = np.exp(-energia(griglia))
esatta /= esatta.sum()
for eps in (0.01, 0.002, 0.0005):
    media = griglia - 0.5 * eps * gradiente(griglia)
    K = np.exp(-(griglia[None, :] - media[:, None])**2 / (2 * eps))
    K[K < 1e-12] = 0.0
    K = sparse.csr_matrix(K / K.sum(1, keepdims=True))
    _, v = eigs(K.T, k=1, which="LM")
    p = np.abs(v[:, 0].real)
    p /= p.sum()
    scarto = p[centro].sum() - esatta[centro].sum()
    print(f"eps = {eps:<7} scarto sul bin centrale {scarto:+.5f}"
          f"   scarto/eps {scarto / eps:.3f}")